# Mission 14 — Evidence-first RAG for the 2024 Year-end Tax Guide

이 노트북은 **검색 평가 → 생성 평가** 순서를 강제합니다. 현재 결과표는 실행 후 채워지는 자리이며, 실행 전 수치를 미리 적지 않습니다.

## Executive Summary (실행 후 갱신)

- 문서: 국세청 「2024년 귀속 연말정산 신고안내」
- 기본 검색: KURE-v1 + Chroma cosine
- 고급 검색: Dense + BM25 → weighted RRF → bge reranker
- 생성: Qwen3-4B-Instruct-2507, 4-bit NF4
- 평가: Hit@1, Hit@5, MRR + 답변 rubric
- 현재 best configuration: **미실행**
- 현재 최종 수치: **미실행**

> 먼저 `data/evaluation_qa.json`의 gold page를 원문과 대조해 `verified`로 바꾼 뒤 검색 점수를 계산합니다.

## 1. Reproducible environment

In [ ]:
from pathlib import Path
import subprocess, sys

REPO_URL = 'https://github.com/uyt5041-lab/mission14.git'
REPO_DIR = Path('/content/mission14')

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-colab.txt')], check=True)
sys.path.insert(0, str(REPO_DIR))
print('Repository and packages are ready:', REPO_DIR)

In [ ]:
import json, os, random, time
import numpy as np
import pandas as pd
import torch, transformers, langchain

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('langchain:', langchain.__version__)
print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Colab 런타임을 GPU로 변경한 뒤 다시 실행하세요.'

## 2. Dataset download and integrity gate

국세청 원문 URL을 기본으로 사용합니다. 다운로드가 차단되면 파일 업로드 창이 열립니다. 미션 제공 URL이나 Drive 경로가 있다면 `SOURCE_URL` 또는 `PDF_PATH`만 바꾸면 됩니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from src.rag_core import OFFICIAL_PDF_URL, validate_pdf

PROJECT_DIR = Path('/content/drive/MyDrive/mission14')
DATA_DIR = PROJECT_DIR / 'data'
INDEX_DIR = PROJECT_DIR / 'indexes'
RESULT_DIR = PROJECT_DIR / 'results'
for directory in (DATA_DIR, INDEX_DIR, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SOURCE_URL = OFFICIAL_PDF_URL
PDF_PATH = DATA_DIR / '2024_year_end_tax_guide.pdf'
print(PDF_PATH)

In [ ]:
import requests

def download_pdf(url, target):
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    if not response.content.startswith(b'%PDF'):
        raise ValueError('응답이 PDF가 아닙니다.')
    target.write_bytes(response.content)

if not PDF_PATH.exists():
    try:
        download_pdf(SOURCE_URL, PDF_PATH)
    except Exception as error:
        print('자동 다운로드 실패:', error)
        from google.colab import files
        uploaded = files.upload()
        uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
        PDF_PATH.write_bytes(uploaded_bytes)
        print('업로드 파일을 저장함:', uploaded_name)

pdf_info = validate_pdf(PDF_PATH)
pdf_info

## 3. Page parsing and conservative cleaning

In [ ]:
from src.rag_core import apply_metadata_rules, load_and_clean_pdf

pages, repeated_lines = load_and_clean_pdf(PDF_PATH)
metadata_rules = json.loads((REPO_DIR / 'data/metadata_rules.json').read_text(encoding='utf-8'))
pages = apply_metadata_rules(pages, metadata_rules)
print('pages:', len(pages))
print('metadata rules:', len(metadata_rules))
print('repeated lines removed:', len(repeated_lines))
print(sorted(repeated_lines)[:30])

In [ ]:
from IPython.display import display, Markdown

SAMPLE_PDF_PAGES = [1, 33, 106, 216]
for page_number in SAMPLE_PDF_PAGES:
    doc = pages[page_number - 1]
    display(Markdown(f'### PDF page {page_number}'))
    print(doc.page_content[:1800])
    print('metadata:', doc.metadata)
    print('-' * 80)

print('STOP GATE: 위 4개 페이지를 PDF 원문과 대조하고 printed_page/section 규칙을 확정하세요.')

## 4. Evaluation QA set — gold evidence first

`gold_status == verified`인 질문만 검색 평가에 들어갑니다. 아래 표에서 원문 확인이 끝난 질문의 page와 expected answer를 먼저 수정합니다. 수정본은 Drive와 저장소에 함께 반영하는 것이 좋습니다.

In [ ]:
QA_PATH = REPO_DIR / 'data/evaluation_qa.json'
qa_rows = json.loads(QA_PATH.read_text(encoding='utf-8'))
qa_df = pd.DataFrame(qa_rows)
display(qa_df[['id', 'type', 'scope', 'question', 'gold_pdf_pages', 'gold_status']])
verified_count = int((qa_df['gold_status'] == 'verified').sum())
print(f'verified gold questions: {verified_count}/{len(qa_df)}')

## 5. Targeted table corrections

표 원문을 확인한 후에만 아래 예시의 placeholder를 실제 값으로 바꾸고 `table_documents`에 추가하세요. 확인 전 값을 vector DB에 넣지 않습니다.

In [ ]:
from src.rag_core import make_table_document

table_documents = []
# 원문 검증 후 주석을 해제하고 정확한 page/value를 입력하세요.
# table_documents.append(make_table_document(
#     item='월세액 세액공제 소득기준 및 한도 상향',
#     previous='원문 확인값', revised='원문 확인값',
#     effective_from='원문 확인값', pdf_page=0,
#     section='2024년 귀속 연말정산 개정세법 요약',
# ))
print('verified table documents:', len(table_documents))

## 6. Token-aware chunking experiments

In [ ]:
from src.rag_core import CHUNK_CONFIGS, chunk_documents
from transformers import AutoTokenizer

source_documents = pages + table_documents
chunks_by_config = {
    name: chunk_documents(source_documents, name)
    for name in CHUNK_CONFIGS
}
for name, chunks in chunks_by_config.items():
    print(name, CHUNK_CONFIGS[name], 'chunks:', len(chunks))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

length_tokenizer = AutoTokenizer.from_pretrained('nlpai-lab/KURE-v1')
length_rows = []
for config_name, chunks in chunks_by_config.items():
    for chunk in chunks:
        length_rows.append({
            'config': config_name,
            'tokens': len(length_tokenizer.encode(chunk.page_content, add_special_tokens=False)),
        })
length_df = pd.DataFrame(length_rows)
display(length_df.groupby('config')['tokens'].describe())
sns.histplot(data=length_df, x='tokens', hue='config', element='step', stat='density', common_norm=False)
plt.title('Chunk token-length distribution')
plt.show()

## 7. Dense indexes and retrieval evaluation

인덱스는 Drive에 저장하고 다시 실행할 때 재사용합니다. Gold page가 하나도 검증되지 않았다면 metric 계산을 의도적으로 중단합니다.

In [ ]:
from src.rag_core import build_embeddings, open_or_build_chroma

embeddings = build_embeddings(device='cuda')
vector_stores = {}
for config_name, chunks in chunks_by_config.items():
    vector_stores[config_name] = open_or_build_chroma(
        chunks,
        INDEX_DIR / f'chroma_kure_{config_name.lower()}',
        collection_name=f'mission14_kure_{config_name.lower()}',
        embeddings=embeddings,
    )
print('indexes ready:', sorted(vector_stores))

In [ ]:
from src.rag_core import dense_search, evaluate_retrieval

verified_rows = [row for row in qa_rows if row['gold_status'] == 'verified']
dense_summaries = {}
if not verified_rows:
    print('STOP GATE: verified gold page가 없습니다. Phase 4를 먼저 완료하세요.')
else:
    for config_name, store in vector_stores.items():
        detail, summary = evaluate_retrieval(
            verified_rows,
            lambda question, s=store: dense_search(s, question, k=5),
            k=5,
        )
        dense_summaries[config_name] = summary
        (RESULT_DIR / f'dense_{config_name.lower()}_detail.json').write_text(
            json.dumps(detail, ensure_ascii=False, indent=2), encoding='utf-8'
        )
    display(pd.DataFrame(dense_summaries).T.sort_values(['hit@5', 'mrr'], ascending=False))

## 8. Hybrid retrieval and reranking

Best chunk는 metric으로 선택합니다. 검증 전에는 C2를 임시 기본값으로 쓸 수 있지만 이를 최종 best라고 보고하지 않습니다.

In [ ]:
from src.rag_core import BM25Index, weighted_rrf, rerank_hits, hits_as_rows
from sentence_transformers import CrossEncoder

BEST_CONFIG = (
    max(dense_summaries, key=lambda name: (dense_summaries[name]['hit@5'], dense_summaries[name]['mrr']))
    if dense_summaries else 'C2'
)
best_chunks = chunks_by_config[BEST_CONFIG]
best_store = vector_stores[BEST_CONFIG]
bm25 = BM25Index(best_chunks)
reranker_model = CrossEncoder('BAAI/bge-reranker-v2-m3', device='cuda')

def hybrid_search(query, use_reranker=True):
    dense_hits = dense_search(best_store, query, k=15)
    bm25_hits = bm25.search(query, k=15)
    fused = weighted_rrf(
        {'dense': dense_hits, 'bm25': bm25_hits},
        {'dense': 0.7, 'bm25': 0.3},
        top_n=12,
    )
    return rerank_hits(query, fused, top_n=5, model=reranker_model) if use_reranker else fused[:5]

demo_question = '2024년 귀속 월세액 세액공제의 총급여 기준과 한도는?'
demo_hits = hybrid_search(demo_question, use_reranker=True)
display(pd.DataFrame(hits_as_rows(demo_hits)))

## 9. Qwen 4-bit + LCEL Basic/Advanced RAG

Embedding 작업이 끝난 뒤 해당 객체를 지우고 Qwen을 로드합니다. T4 메모리가 부족하면 인덱스를 저장한 뒤 런타임을 재시작하고 Chroma만 다시 여세요.

In [ ]:
# 생성할 모든 질문의 검색 결과를 먼저 고정해 두면 검색 모델을 내려도 됩니다.
questions_for_generation = list(dict.fromkeys([demo_question] + [row['question'] for row in qa_rows]))
precomputed_hits = {question: hybrid_search(question) for question in questions_for_generation}

import gc
del embeddings, vector_stores, best_store, reranker_model
gc.collect()
torch.cuda.empty_cache()
print('GPU cache cleared before loading the generator.')

In [ ]:
from src.rag_core import build_qwen_lcel_chain, answer_with_evidence

generation_chain = build_qwen_lcel_chain()
result = answer_with_evidence(generation_chain, demo_question, precomputed_hits[demo_question])
display(pd.DataFrame(result['evidence']))
print(result['answer'])

## 10. Final evaluation and failure analysis

아래 산출물을 채운 뒤 Executive Summary를 갱신합니다.

- Dense C1/C2/C3: Hit@1, Hit@5, MRR, latency
- Best Dense vs Hybrid vs Hybrid+Rerank
- E0 No RAG vs E1 Basic RAG vs E4 Advanced RAG
- 범위 밖 질문 거절률
- 실패 사례 2개: 원인 → 수정 → metric/answer 변화

### 답변 rubric (각 문항)

| 항목 | 0점 | 1점 | 2점 |
|---|---|---|---|
| 정확성 | 틀림 | 일부 정확 | 정확 |
| 근거 충실성 | 근거 없음 | 일부 근거 | 전부 문서 기반 |
| 조건·예외 | 누락 | 일부 포함 | 핵심 포함 |
| 완전성 | 핵심 누락 | 대체로 답함 | 충분히 답함 |

Citation 정확성 0~1, 문서 밖 질문 거절 0~1을 더해 총 10점으로 기록합니다.